# Week 36 — Data audit and rule-based baselines

Reliable Multilingual QA (DIKU NLP 2026). All logic lives in `src/`, so the same code is reused unchanged in weeks 37–44

**Covers:** Section 2 Items 1–3 of the project description, plus the two
mandatory answerability baselines, all scored under the Section 1 protocol.

Every table is also written to `results/` as `.csv` and `.md`.

## 0. Setup

Pick **one** of the three options below. On Colab, `drive` is the least painful
because results survive a runtime restart.

In [ ]:
SETUP = "local"  # "local" | "drive" | "upload"
PROJECT_DIR = None

import os, sys, subprocess

if SETUP == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/nlp-project'   # <- adjust
elif SETUP == "upload":
    from google.colab import files
    up = files.upload()                                   # upload nlp-project.zip
    name = next(iter(up))
    subprocess.run(['unzip', '-oq', name, '-d', '/content'], check=True)
    PROJECT_DIR = '/content/nlp-project'
else:
    PROJECT_DIR = os.path.abspath('..')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('project dir:', PROJECT_DIR)
print(sorted(os.listdir(PROJECT_DIR)))

In [ ]:
!pip -q install "datasets>=2.19" pandas numpy scikit-learn tabulate

In [ ]:
import importlib
import pandas as pd

from src import audit, baselines, config, data, evaluation, reporting, tokenisation
for m in (config, tokenisation, data, evaluation, audit, baselines, reporting):
    importlib.reload(m)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
print('languages:', config.LANGS, '| splits:', config.SPLITS, '| seed:', config.SEED)

### Sanity check the pipeline before touching the real data

Runs the self-tests against a synthetic fixture containing every edge case the
project asks us to handle (answer at char 0, multi-token answer, adjacent
punctuation, repeated answer string, unanswerable rows, a deliberately broken
offset). All of these must pass.

In [ ]:
!python tests/test_week36.py

## 1. Load, and justify the language filter

The published dataset contains more question languages than the project uses.
Report how many rows the ar/ko/te filter discards — every later count is
post-filter.

In [ ]:
raw = data.load_raw()
inventory = data.language_inventory(raw)
reporting.save_table(inventory, '00_language_inventory')
inventory

In [ ]:
splits = data.load_project_splits(raw)
{k: len(v) for k, v in splits.items()}

## 2. Item 1 — counts, proportions, length distributions, missing values, duplicates

In [ ]:
desc = pd.concat([audit.describe_split(df, s) for s, df in splits.items()], ignore_index=True)
reporting.save_table(desc, '01_item1_describe')
desc[['split','lang','n','n_answerable','n_unanswerable','prop_answerable',
      'q_len_median','q_len_iqr','ctx_len_median','ctx_len_iqr',
      'dup_pair_groups','dup_pair_redundant_rows']]

In [ ]:
miss = pd.concat([audit.missing_values(df, s) for s, df in splits.items()], ignore_index=True)
reporting.save_table(miss, '02_item1_missing_values')
miss

In [ ]:
for split, df in splits.items():
    d = audit.duplicate_pairs(df)
    print(split, d, 'label-inconsistent groups:', audit.label_inconsistent_duplicates(df))
print('cross-split overlap:', audit.cross_split_overlap(splits))
audit.duplicate_pair_examples(splits['train'])

### Tokenisation

`src/tokenisation.py` explains why neither `str.split()` nor `re.findall(r"\w+")`
is acceptable here. The cell below is the demonstration to quote in the report:
`\w+` fragments Telugu and vocalised Arabic words because Unicode combining
marks are not alphanumeric.

In [ ]:
tokenisation.run_selftest()
tok = audit.tokeniser_comparison(splits['train'])
reporting.save_table(tok, '04_tokeniser_comparison')
tok

## 3. Item 2 — five most common question tokens per language (training split only)

Any token flagged `translation_verified == False` is **not** in the glossary and
must be translated by hand before it goes in the report.

In [ ]:
top = audit.top_question_tokens(splits['train'], k=5)
reporting.save_table(top, '05_item2_top_question_tokens')
display(top)
todo = top[~top['translation_verified']]
print(f'{len(todo)} token(s) need a hand translation')
todo[['lang','token','count']]

## 4. Item 3 — `answer` vs `context[answer_start:]`

In [ ]:
for split, df in splits.items():
    res = audit.verify_answer_offsets(df, split)
    print(f"{split}: checked {res['n_answerable_checked']}, failures {res['n_failures']}")
    print('   per language:', res['per_language'])
    print('   unanswerable violating (-1, ""):', res['n_unanswerable_violating_convention'])
    print('   answers occurring >1x in context:', res['n_answers_occurring_more_than_once'])
    if len(res['failures']):
        display(res['failures'].head(10))

## 5. The two baselines, under the Section 1 protocol

In [ ]:
maj = baselines.MajorityClassBaseline().fit(splits['train'])
rule = baselines.CrossLingualAnchorRule().fit(splits['train'])
print(maj.describe())
print(rule.describe())

In [ ]:
diag = rule.anchor_diagnostics(splits['train'])
reporting.save_table(diag, '07_anchor_diagnostics_train')
display(diag)
sweep = rule.sweep_min_hits(splits['train'])
reporting.save_table(sweep, '08_anchor_min_hits_sweep_train')
sweep[['min_hits','n','macro_f1','answerable_f1','unanswerable_f1','accuracy']]

In [ ]:
models = {
    'majority-class': maj,
    'anchor-rule': rule,
    'anchor-rule (label-free fallback)': baselines.CrossLingualAnchorRule(fallback=True).fit(splits['train']),
}
reports = []
for split, df in splits.items():
    for name, model in models.items():
        rep = evaluation.answerability_report(df, model.predict(df), name)
        rep.insert(1, 'split', split)
        reports.append(rep)
results = pd.concat(reports, ignore_index=True)
reporting.save_table(results, '10_week36_answerability_results')
results[results.split == 'validation'][
    ['model','subset','n','answerable_f1','unanswerable_f1','macro_f1','accuracy']
]

In [ ]:
print(reporting.save_environment(extra={'notebook': 'week36_data_and_baselines'}))

## 6. What to look at before writing

- Does the anchor rule beat majority on **unanswerable-class F1**? Macro-F1 alone
  can hide a rule that never predicts the minority class.
- Does `prop_questions_with_any_anchor` differ a lot across ar/ko/te? That is the
  quantitative version of the "what the rule cannot exploit" argument.
- Are there label-inconsistent duplicates? If so, they cap every model's ceiling
  and belong in limitations.
- Is there any train/validation overlap? If so, say so — it inflates everything.